# Basic Code to test HDMI

- First test of creating the Mandelbrot Set and displaying it on a monitor using the default base overlay

In [ ]:
# load the stock base overlay
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import VideoMode, PIXEL_RGB
import numpy as np

base = BaseOverlay("base.bit")     # the default base shipped with PYNQ
hdmi_out = base.video.hdmi_out

mode = VideoMode(1280, 720, 24)     # 640x480 or 1280x720
hdmi_out.configure(mode, PIXEL_RGB)
hdmi_out.start()

In [ ]:
# Mandelbrot in vectorised NumPy + colour mapping
import numpy as np
import time
import matplotlib.cm as cm

def mandelbrot_numpy(width, height, center_r, center_i, zoom, max_iter=256):
    """Returns (H, W) uint16 escape counts. max_iter = inside the set."""
    view_w = 4.0 / zoom
    view_h = view_w * height / width
    x = np.linspace(center_r - view_w/2, center_r + view_w/2, width,  dtype=np.float32)
    y = np.linspace(center_i - view_h/2, center_i + view_h/2, height, dtype=np.float32)
    Cr, Ci = np.meshgrid(x, y)
    Zr = np.zeros_like(Cr)
    Zi = np.zeros_like(Ci)
    iter_count = np.full(Cr.shape, max_iter, dtype=np.uint16)
    active = np.ones(Cr.shape, dtype=bool)

    for n in range(max_iter):
        Zr2 = Zr * Zr
        Zi2 = Zi * Zi
        escaped = active & (Zr2 + Zi2 > 4.0)
        iter_count[escaped] = n
        active &= ~escaped
        # update z = z^2 + c
        Zi = 2.0 * Zr * Zi + Ci
        Zr = Zr2 - Zi2 + Cr
    return iter_count

def colourise(iter_count, max_iter, cmap_name='twilight_shifted'):
    """Returns (H, W, 3) uint8 RGB."""
    t = iter_count.astype(np.float32) / max_iter
    rgba = cm.get_cmap(cmap_name)(t)
    rgb = (rgba[..., :3] * 255).astype(np.uint8)
    rgb[iter_count == max_iter] = 0  # inside-set pixels black
    return rgb

In [ ]:
# render, time it, push to HDMI
W, H = 1280, 720           # 640x480 or 1280x720
MAX_ITER = 200

t0 = time.perf_counter()
escape = mandelbrot_numpy(W, H, center_r=-0.5, center_i=0.0, zoom=1.0, max_iter=MAX_ITER)
t_render = time.perf_counter() - t0

t0 = time.perf_counter()
rgb = colourise(escape, MAX_ITER)
t_colour = time.perf_counter() - t0

print(f"Render: {t_render*1000:7.0f} ms   ({(W*H/1e6)/t_render:.2f} Mpix/s)")
print(f"Colour: {t_colour*1000:7.0f} ms")

frame = hdmi_out.newframe()
frame[...] = rgb           # in-place — never reassign frame
hdmi_out.writeframe(frame)

/tmp/ipykernel_896/1724060588.py:19: RuntimeWarning: overflow encountered in multiply
  Zr2 = Zr * Zr
/tmp/ipykernel_896/1724060588.py:20: RuntimeWarning: overflow encountered in multiply
  Zi2 = Zi * Zi
/tmp/ipykernel_896/1724060588.py:21: RuntimeWarning: overflow encountered in add
  escaped = active & (Zr2 + Zi2 > 4.0)
/tmp/ipykernel_896/1724060588.py:25: RuntimeWarning: overflow encountered in multiply
  Zi = 2.0 * Zr * Zi + Ci
/tmp/ipykernel_896/1724060588.py:26: RuntimeWarning: invalid value encountered in subtract
  Zr = Zr2 - Zi2 + Cr


Render:   66116 ms   (0.01 Mpix/s)
Colour:     985 ms


In [ ]:
# clean up when done
hdmi_out.stop()
del hdmi_out

Display on PYNQ board:

![Mandelbrot on PYNQ](../docs/assets/simple_mandelbrot_set_visual.jpeg)

Render:   66116 ms   (0.01 Mpix/s)
Colour:     985 ms